In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2009_2010"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "HUQ_F.xpt"
csv_file = DATA_DIR / "HUQ_F_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve untouched pandas-decoded XPT values.
#
source_df = df.copy()


print("=" * 75)
print("NHANES 2009-2010 HUQ_F XPT -> CSV")
print("=" * 75)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "HUQ010",
    "HUQ020",
    "HUQ030",
    "HUQ040",
    "HUQ050",
    "HUQ060",
    "HUQ071",
    "HUD080",
    "HUQ090"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "HUQ_F column names/order differ from expected.\n"
        f"Actual: {df.columns.tolist()}"
    )


if len(df) != 10537:

    raise ValueError(
        f"Unexpected row count: {len(df)}"
    )


if len(df.columns) != 10:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 10,537 rows x 10 columns."
)


# ============================================================
# 4. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT zero-decoding artifact:
#
#     5.397605346934028e-79
#
# IMPORTANT:
#
# Do NOT globally replace this value.
#
# Validate:
#
#     variable
#     count
#     CDC codebook meaning
#
# before changing anything.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_total = int(
    df[numeric_cols]
    .eq(0)
    .sum()
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")


print(
    "True numeric zeros:",
    f"{true_zero_total:,}"
)


print(
    "Tiny artifact occurrences:",
    f"{tiny_total:,}"
)


if tiny_total > 0:

    print(
        "\nTiny values by variable:"
    )

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


# ============================================================
# 5. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Attached HUQ_F contains exactly:
#
#     HUQ050 = 1,398 tiny values
#
# CDC:
#
# HUQ050 = # times receive healthcare over past year
#
# 0 = None
#
# Published count:
#
# 0 = 1,398
#
# Therefore these 1,398 tiny values are validated zeros.
#

EXPECTED_TINY_COUNTS = {
    "HUQ050": 1398
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated HUQ_F pattern.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {actual_tiny_counts}"
    )


if tiny_total != 1398:

    raise ValueError(
        f"Unexpected total tiny-value count: {tiny_total}"
    )


print(
    "\nPASS: Exact HUQ050 tiny-zero pattern validated."
)


# ============================================================
# 6. RESTORE VALIDATED HUQ050 ZEROS
# ============================================================

#
# Restore ONLY the validated HUQ050 values.
#
# No global tiny-value replacement.
#

tiny_mask = (
    df["HUQ050"]
    .eq(TINY_VALUE)
)


correction_count = int(
    tiny_mask.sum()
)


if correction_count != 1398:

    raise ValueError(
        "Unexpected HUQ050 tiny-value count."
    )


df.loc[
    tiny_mask,
    "HUQ050"
] = 0


print("\n--- ZERO RESTORATION ---")


print(
    "HUQ050 values restored to zero:",
    correction_count
)


if int(
    df["HUQ050"]
    .eq(0)
    .sum()
) != 1398:

    raise ValueError(
        "HUQ050 zero restoration failed."
    )


print(
    "PASS: 1,398 validated HUQ050 zeros restored."
)


# ============================================================
# 7. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

#
# expected_df represents exactly what the corrected
# dataset should contain.
#
# The ONLY expected difference from source_df:
#
#     HUQ050 tiny -> 0
#

expected_df = source_df.copy()


expected_df.loc[
    expected_df["HUQ050"].eq(TINY_VALUE),
    "HUQ050"
] = 0


# ============================================================
# 8. CHECK FOR GENUINE DECIMAL VALUES
# ============================================================

#
# CRITICAL FIDELITY RULE:
#
# Never round genuine decimal values.
#
# Always perform this test BEFORE Int64 conversion.
#
# HUQ_F contains no genuine fractional variables.
#
# This same safeguard protects:
#
#     Dietary decimal values
#     Lab measurements
#     Examination measurements
#     Sample weights
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if fractional_mask.any():

        fractional_columns.append(
            col
        )

        print(
            f"{col}: "
            f"{int(fractional_mask.sum()):,} "
            "genuine fractional values"
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal values detected. "
        "They will NOT be rounded:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional values "
    "exist in HUQ_F."
)


# ============================================================
# 9. CLEAN AND VALIDATE SEQN
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


if (
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
    != 0
    or
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
    != 0
    or
    df["SEQN"].min()
    != 51624
    or
    df["SEQN"].max()
    != 62160
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 10. FORMAT QUESTIONNAIRE VALUES WITHOUT .0
# ============================================================

#
# After:
#
#     validated zero restoration
#     fractional-value check
#
# all HUQ_F variables are proven whole-number fields.
#
# Therefore Int64 conversion is safe.
#
# NO .round() is used.
#

for col in df.columns:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_number_check = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not np.all(
            whole_number_check
        ):

            raise ValueError(
                f"{col} contains genuine decimal values "
                "and cannot safely be converted to Int64."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All HUQ_F fields formatted "
    "without unnecessary .0."
)


# ============================================================
# 11. CDC CHECK - HUQ010
# ============================================================

#
# General health condition
#
# CDC:
#
# 1 = Excellent     2,594
# 2 = Very good     2,718
# 3 = Good          3,384
# 4 = Fair          1,528
# 5 = Poor            310
# 7 = Refused           0
# 9 = Don't know        3
# Missing               0
#

EXPECTED_HUQ010 = {
    1: 2594,
    2: 2718,
    3: 3384,
    4: 1528,
    5: 310,
    9: 3
}


actual = (
    df["HUQ010"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_HUQ010
    or
    int(
        df["HUQ010"]
        .isna()
        .sum()
    )
    != 0
):

    raise ValueError(
        "HUQ010 differs from CDC."
    )


print(
    "PASS: HUQ010 matches CDC."
)


# ============================================================
# 12. CDC CHECK - HUQ020
# ============================================================

#
# Health now compared with one year ago
#
# CDC:
#
# 1 = Better          1,793
# 2 = Worse             843
# 3 = About same      7,472
# 7 = Refused             1
# 9 = Don't know          0
# Missing               428
#

EXPECTED_HUQ020 = {
    1: 1793,
    2: 843,
    3: 7472,
    7: 1
}


actual = (
    df["HUQ020"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_HUQ020
    or
    int(
        df["HUQ020"]
        .isna()
        .sum()
    )
    != 428
):

    raise ValueError(
        "HUQ020 differs from CDC."
    )


print(
    "PASS: HUQ020 matches CDC."
)


# ============================================================
# 13. CDC CHECK - HUQ030
# ============================================================

#
# Routine place to go for healthcare
#
# CDC:
#
# 1 = Yes                         9,258
# 2 = No routine place            1,239
# 3 = More than one place            38
# 7 = Refused                         1
# 9 = Don't know                      1
# Missing                             0
#

EXPECTED_HUQ030 = {
    1: 9258,
    2: 1239,
    3: 38,
    7: 1,
    9: 1
}


actual = (
    df["HUQ030"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_HUQ030
    or
    int(
        df["HUQ030"]
        .isna()
        .sum()
    )
    != 0
):

    raise ValueError(
        "HUQ030 differs from CDC."
    )


print(
    "PASS: HUQ030 matches CDC."
)


# ============================================================
# 14. CDC CHECK - HUQ040
# ============================================================

#
# Type of place most often used for healthcare
#
# CDC:
#
# 1 = Clinic / health center             2,484
# 2 = Doctor's office / HMO              6,287
# 3 = Hospital emergency room              278
# 4 = Hospital outpatient department       155
# 5 = Some other place                       90
# 7 = Refused                                 0
# 9 = Don't know                              2
# Missing                                 1,241
#

EXPECTED_HUQ040 = {
    1: 2484,
    2: 6287,
    3: 278,
    4: 155,
    5: 90,
    9: 2
}


actual = (
    df["HUQ040"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_HUQ040
    or
    int(
        df["HUQ040"]
        .isna()
        .sum()
    )
    != 1241
):

    raise ValueError(
        "HUQ040 differs from CDC."
    )


print(
    "PASS: HUQ040 matches CDC."
)


# ============================================================
# 15. CDC CHECK - HUQ050
# ============================================================

#
# Number of times received healthcare over past year
#
# CDC:
#
# 0 = None          1,398
# 1 = 1             2,039
# 2 = 2 to 3        3,237
# 3 = 4 to 9        2,626
# 4 = 10 to 12        635
# 5 = 13 or more      587
# 77 = Refused          1
# 99 = Don't know      14
# Missing               0
#
# The 1,398 raw tiny XPORT values correspond exactly
# to CDC code 0.
#

EXPECTED_HUQ050 = {
    0: 1398,
    1: 2039,
    2: 3237,
    3: 2626,
    4: 635,
    5: 587,
    77: 1,
    99: 14
}


actual = (
    df["HUQ050"]
    .value_counts()
    .sort_index()
    .to_dict()
)


print("\n--- HUQ050 ---")


print(
    actual
)


if (
    actual != EXPECTED_HUQ050
    or
    int(
        df["HUQ050"]
        .isna()
        .sum()
    )
    != 0
):

    raise ValueError(
        "HUQ050 differs from CDC / "
        "validated zero restoration."
    )


if int(
    df["HUQ050"]
    .eq(0)
    .sum()
) != 1398:

    raise ValueError(
        "HUQ050 zero restoration count is incorrect."
    )


print(
    "PASS: HUQ050 matches CDC."
)


# ============================================================
# 16. CDC CHECK - HUQ060
# ============================================================

#
# How long since last healthcare visit
#
# CDC:
#
# 1 = 6 months or less                   50
# 2 = >6 months <=1 year                 88
# 3 = >1 year <=3 years                 799
# 4 = >3 years                          427
# 5 = Never                              32
# 7 = Refused                             0
# 9 = Don't know                          2
# Missing                             9,139
#

EXPECTED_HUQ060 = {
    1: 50,
    2: 88,
    3: 799,
    4: 427,
    5: 32,
    9: 2
}


actual = (
    df["HUQ060"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_HUQ060
    or
    int(
        df["HUQ060"]
        .isna()
        .sum()
    )
    != 9139
):

    raise ValueError(
        "HUQ060 differs from CDC."
    )


print(
    "PASS: HUQ060 matches CDC."
)


# ============================================================
# 17. CDC CHECK - HUQ071
# ============================================================

#
# Overnight hospital patient in last year
#
# CDC:
#
# 1 = Yes          1,088
# 2 = No           9,447
# 7 = Refused          1
# 9 = Don't know       1
# Missing              0
#

EXPECTED_HUQ071 = {
    1: 1088,
    2: 9447,
    7: 1,
    9: 1
}


actual = (
    df["HUQ071"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_HUQ071
    or
    int(
        df["HUQ071"]
        .isna()
        .sum()
    )
    != 0
):

    raise ValueError(
        "HUQ071 differs from CDC."
    )


print(
    "PASS: HUQ071 matches CDC."
)


# ============================================================
# 18. CDC CHECK - HUD080
# ============================================================

#
# Number of overnight hospitalizations in past year
#
# CDC:
#
# 1-5 = Range of values       1,070
# 6   = 6 times or more          18
# 77777 = Refused                 0
# 99999 = Don't know              0
# Missing                     9,449
#

s = df["HUD080"]


print("\n--- HUD080 ---")


print(
    "1-5:",
    int(
        s.between(
            1,
            5
        ).sum()
    )
)


print(
    "6 or more:",
    int(
        s.eq(6)
        .sum()
    )
)


print(
    "Missing:",
    int(
        s.isna()
        .sum()
    )
)


if not (
    int(
        s.between(
            1,
            5
        ).sum()
    )
    == 1070
    and
    int(
        s.eq(6)
        .sum()
    )
    == 18
    and
    int(
        s.eq(77777)
        .sum()
    )
    == 0
    and
    int(
        s.eq(99999)
        .sum()
    )
    == 0
    and
    int(
        s.isna()
        .sum()
    )
    == 9449
):

    raise ValueError(
        "HUD080 differs from CDC."
    )


print(
    "PASS: HUD080 matches CDC."
)


# ============================================================
# 19. CDC CHECK - HUQ090
# ============================================================

#
# Seen mental health professional in past year
#
# CDC:
#
# 1 = Yes            702
# 2 = No           8,553
# 7 = Refused          1
# 9 = Don't know       2
# Missing          1,279
#

EXPECTED_HUQ090 = {
    1: 702,
    2: 8553,
    7: 1,
    9: 2
}


actual = (
    df["HUQ090"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_HUQ090
    or
    int(
        df["HUQ090"]
        .isna()
        .sum()
    )
    != 1279
):

    raise ValueError(
        "HUQ090 differs from CDC."
    )


print(
    "PASS: HUQ090 matches CDC."
)


# ============================================================
# 20. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# Expected change from raw pandas XPT:
#
# HUQ050:
#
#     1,398 TINY_VALUE values -> numeric 0
#
# NOTHING ELSE may change.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        failures.append(
            col
        )


if failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(failures)
    )


print(
    "PASS: All 10 variables match expected "
    "corrected XPT values exactly."
)


# ============================================================
# 21. EXPORT CSV
# ============================================================

#
# IMPORTANT:
#
# NO float_format.
# NO rounding.
#
# All HUQ_F variables are proven whole-number fields.
#
# Missing values become blank.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 22. RE-READ EXPORTED CSV
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 23. EXACT CORRECTED XPT -> CSV FIDELITY
# ============================================================

print("\n--- XPT -> CSV FIDELITY ---")


csv_failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV differences found in:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 10 CSV columns match "
    "expected corrected XPT values exactly."
)


# ============================================================
# 24. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in expected_df.columns:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE CHECK ---")


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 25. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in csv_df.select_dtypes(
    include=[np.number]
).columns:

    tiny_csv_count += int(
        csv_df[col]
        .eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain in CSV."
    )


print(
    "PASS: No tiny-value artifacts remain."
)


# ============================================================
# 26. RAW CSV .0 FORMAT CHECK
# ============================================================

#
# HUQ_F contains NO genuine decimal fields.
#
# Therefore ".0" should not occur anywhere in this
# particular final CSV.
#
# DO NOT use this global test for Dietary or other
# files containing genuine decimals.
#

print("\n--- RAW CSV FORMAT CHECK ---")


with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


dot_zero_count = (
    csv_text.count(".0")
)


print(
    "Occurrences of '.0':",
    dot_zero_count
)


if dot_zero_count != 0:

    raise ValueError(
        "Unexpected .0 formatting remains."
    )


print(
    "PASS: No unnecessary .0 formatting."
)


# ============================================================
# 27. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 28. FINAL CSV SEQN CHECK
# ============================================================

print("\n--- FINAL CSV SEQN CHECK ---")


if (
    csv_df["SEQN"].min()
    != 51624
    or
    csv_df["SEQN"].max()
    != 62160
    or
    int(
        csv_df["SEQN"]
        .isna()
        .sum()
    )
    != 0
    or
    int(
        csv_df["SEQN"]
        .duplicated()
        .sum()
    )
    != 0
):

    raise ValueError(
        "Final CSV SEQN validation failed."
    )


print(
    "PASS: Final CSV SEQN is valid."
)


# ============================================================
# 29. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 75
)


print(
    "HUQ_F CONVERSION COMPLETE"
)


print(
    "=" * 75
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)


print(
    "HUQ050 zero restorations:",
    correction_count
)


print(
    "Genuine decimal variables: NONE"
)


print(
    "XPT -> CSV unexpected differences: 0 expected"
)


print(
    "Missing-value differences: 0 expected"
)


print(
    "Tiny values remaining: 0 expected"
)


print(
    "Unnecessary .0: 0 expected"
)


print(
    "Scientific notation: 0 expected"
)

NHANES 2009-2010 HUQ_F XPT -> CSV
Rows: 10,537
Columns: 10
PASS: Structure = 10,537 rows x 10 columns.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact occurrences: 1,398

Tiny values by variable:
HUQ050    1398

PASS: Exact HUQ050 tiny-zero pattern validated.

--- ZERO RESTORATION ---
HUQ050 values restored to zero: 1398
PASS: 1,398 validated HUQ050 zeros restored.

--- FRACTIONAL VALUE CHECK ---
PASS: No genuine fractional values exist in HUQ_F.

--- SEQN ---
Missing: 0
Duplicates: 0
Minimum: 51624
Maximum: 62160
PASS: SEQN validated.

PASS: All HUQ_F fields formatted without unnecessary .0.
PASS: HUQ010 matches CDC.
PASS: HUQ020 matches CDC.
PASS: HUQ030 matches CDC.
PASS: HUQ040 matches CDC.

--- HUQ050 ---
{np.int64(0): 1398, np.int64(1): 2039, np.int64(2): 3237, np.int64(3): 2626, np.int64(4): 635, np.int64(5): 587, np.int64(77): 1, np.int64(99): 14}
PASS: HUQ050 matches CDC.
PASS: HUQ060 matches CDC.
PASS: HUQ071 matches CDC.

--- HUD080 ---
1-5: 1070
6 or mo